In [1]:
import numpy as np
import pandas as pd
from tensorflow.keras.datasets import mnist # mnist 훈련셋과 테스트셋
from tensorflow.keras.utils import to_categorical # 원핫인코딩
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout
from matplotlib import pyplot as plt # 학습과정 loss와 acc 시각화
# quiz에서는 scale조정, train_test_split 등을 추가
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
import os
from tensorflow.keras.callbacks import EarlyStopping, Callback, ModelCheckpoint
from tensorflow.keras.models import load_model, save_model

In [34]:
data = pd.read_csv('data/personality_dataset.csv')
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2900 entries, 0 to 2899
Data columns (total 8 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Time_spent_Alone           2837 non-null   float64
 1   Stage_fear                 2827 non-null   object 
 2   Social_event_attendance    2838 non-null   float64
 3   Going_outside              2834 non-null   float64
 4   Drained_after_socializing  2848 non-null   object 
 5   Friends_circle_size        2823 non-null   float64
 6   Post_frequency             2835 non-null   float64
 7   Personality                2900 non-null   object 
dtypes: float64(5), object(3)
memory usage: 181.4+ KB


In [12]:
data.isnull().sum()

Time_spent_Alone             63
Stage_fear                   73
Social_event_attendance      62
Going_outside                66
Drained_after_socializing    52
Friends_circle_size          77
Post_frequency               65
Personality                   0
dtype: int64

In [19]:
data[data.isna().any(axis=1)]

,Time_spent_Alone,Stage_fear,Social_event_attendance,Going_outside,Drained_after_socializing,Friends_circle_size,Post_frequency,Personality
6,4.0,No,9.0,NaN,No,7.0,7.0,Extrovert
33,8.0,Yes,3.0,3.0,NaN,2.0,0.0,Introvert
41,1.0,No,4.0,5.0,No,NaN,10.0,Extrovert
59,NaN,No,9.0,4.0,No,7.0,5.0,Extrovert
71,1.0,No,NaN,5.0,No,11.0,8.0,Extrovert
...,...,...,...,...,...,...,...,...
2882,1.0,NaN,9.0,3.0,No,7.0,6.0,Extrovert
2885,10.0,Yes,0.0,0.0,NaN,2.0,0.0,Introvert
2893,9.0,NaN,2.0,0.0,Yes,4.0,2.0,Introvert
2894,0.0,No,9.0,3.0,No,12.0,NaN,Extrovert


In [39]:
data.fillna(value=data.median(), inplace=True)
data.isna().sum()

/var/folders/xv/plpjkvp94hz67l4s_29k_hqr0000gn/T/ipykernel_1058/1229650355.py:1: FutureWarning: The default value of numeric_only in DataFrame.median is deprecated. In a future version, it will default to False. In addition, specifying 'numeric_only=None' is deprecated. Select only valid columns or specify the value of numeric_only to silence this warning.
  data.fillna(value=data.median(), inplace=True)


Time_spent_Alone             0
Stage_fear                   0
Social_event_attendance      0
Going_outside                0
Drained_after_socializing    0
Friends_circle_size          0
Post_frequency               0
Personality                  0
dtype: int64

In [22]:
num_cols = data.select_dtypes(include=['int64', 'float64']).columns
cat_cols = data.select_dtypes(include=['object', 'category', 'bool']).columns

In [31]:
print(num_cols)
print(cat_cols)

Index(['Time_spent_Alone', 'Social_event_attendance', 'Going_outside',
       'Friends_circle_size', 'Post_frequency'],
      dtype='object')
Index(['Stage_fear', 'Drained_after_socializing', 'Personality'], dtype='object')


In [33]:
data.isna().sum()

Time_spent_Alone              0
Stage_fear                   73
Social_event_attendance       0
Going_outside                 0
Drained_after_socializing    52
Friends_circle_size           0
Post_frequency                0
Personality                   0
dtype: int64

In [37]:
for col in cat_cols:
    most_frequent_value = data[col].mode()[0]
    data[col].fillna(most_frequent_value, inplace=True)
    print(f"컬럼 '{col}' 대체 후 결측치 개수: {data[col].isnull().sum()}개")

컬럼 'Stage_fear' 대체 후 결측치 개수: 0개
컬럼 'Drained_after_socializing' 대체 후 결측치 개수: 0개
컬럼 'Personality' 대체 후 결측치 개수: 0개


In [40]:
data.isna().sum()

Time_spent_Alone             0
Stage_fear                   0
Social_event_attendance      0
Going_outside                0
Drained_after_socializing    0
Friends_circle_size          0
Post_frequency               0
Personality                  0
dtype: int64

In [53]:
X_data = data.iloc[:, :-1]
y_data = data.iloc[:, -1]
X_data.shape, y_data.shape

((2900, 7), (2900,))

In [61]:
X_encoded = pd.get_dummies(X_data, drop_first=True)
# X_encoded
X_encoded

,Time_spent_Alone,Social_event_attendance,Going_outside,Friends_circle_size,Post_frequency,Stage_fear_Yes,Drained_after_socializing_Yes
0,4.0,4.0,6.0,13.0,5.0,0,0
1,9.0,0.0,0.0,0.0,3.0,1,1
2,9.0,1.0,2.0,5.0,2.0,1,1
3,0.0,6.0,7.0,14.0,8.0,0,0
4,3.0,9.0,4.0,8.0,5.0,0,0
...,...,...,...,...,...,...,...
2895,3.0,7.0,6.0,6.0,6.0,0,0
2896,3.0,8.0,3.0,14.0,9.0,0,0
2897,4.0,1.0,1.0,4.0,0.0,1,1
2898,11.0,1.0,3.0,2.0,0.0,1,1
